# Advanced Predictive Maintenance: Turbofan Engine Degradation (NASA C-MAPSS)

## 9. Results and Comparative Graphs

In [ ]:
display(df_results.sort_values(by='Macro F1', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.set_style('whitegrid')

# Plot Macro F1
sns.barplot(data=df_results.sort_values('Macro F1', ascending=False), x='Macro F1', y='Model', palette='viridis', ax=axes[0])
axes[0].set_title('Macro F1 Score (Higher is better)', fontsize=12)
axes[0].set_xlabel('Macro F1')
axes[0].set_xlim(0.6, 1.0)

# Plot Accuracy
sns.barplot(data=df_results.sort_values('Accuracy', ascending=False), x='Accuracy', y='Model', palette='magma', ax=axes[1])
axes[1].set_title('Accuracy General (Bigger is better)', fontsize=12)
axes[1].set_xlabel('Accuracy')
axes[1].set_xlim(0.6, 1.0)
axes[1].set_ylabel('')

# Plot Time
sns.barplot(data=df_results.sort_values('Time (s)'), x='Time (s)', y='Model', palette='cool', ax=axes[2])
axes[2].set_title('Training Time (Smaller is better)', fontsize=12)
axes[2].set_xlabel('Seconds')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

# =========================================================================
# ISOLATED ANALYSIS OF THE BEST MODEL
# =========================================================================
best_model_name = df_results.loc[df_results['Macro F1'].idxmax(), 'Model']
print(f"\n The overall winner of the benchmark is:**{best_model_name}**")
best_preds = predictions_dict[best_model_name]['y_pred']
best_probs = predictions_dict[best_model_name]['y_probs']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Confusion Matrix
cm = confusion_matrix(y_true, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_y.classes_, yticklabels=le_y.classes_, ax=axes[0])
axes[0].set_xlabel('Model Prediction')
axes[0].set_ylabel('Actual Condition')
axes[0].set_title(f'Confusion Matrix: {best_model_name}', fontsize=13)

# 2. Multiclass ROC Curve (One-vs-Rest)
y_test_bin = label_binarize(y_true, classes=[0, 1, 2])
colors = ['#27ae60', '#f39c12', '#c0392b']
for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], best_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, lw=2, color=colors[i], label=f'{le_y.inverse_transform([i])[0]} (AUC={roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.5)
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].set_title(f"ROC Curve (One-vs-Rest): {best_model_name}", fontsize=13)
axes[1].legend(loc="lower right")

sns.despine()
plt.tight_layout()
plt.show()

print(f"\nDetailed Report ({best_model_name}):")
print(classification_report(y_true, best_preds, target_names=le_y.classes_))